In [41]:
import os
import re
import json
import base64
from io import StringIO
from typing import List
from collections import Counter

from unstructured.staging.base import elements_to_json, elements_from_json
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

import ipywidgets as widgets
from IPython.display import display, HTML, Image as IPImage, clear_output

import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import camelot
import pikepdf

load_dotenv()
plt.rcParams['font.family'] = 'DejaVu Sans'

os.makedirs("extracted_data/images", exist_ok=True)
os.makedirs("extracted_data/tables", exist_ok=True)

In [42]:
def remove_pdf_restrictions(input_path, output_path=None):
    """Strip extraction-restriction flags from a PDF (some PDFs disable text extraction)"""
    if output_path is None:
        output_path = input_path.replace(".pdf", "_unlocked.pdf")
    with pikepdf.open(input_path, allow_overwriting_input=True) as pdf:
        pdf.save(output_path)
    print(f"✅ Saved unrestricted copy to {output_path}")
    return output_path

In [43]:
def partition_document(file_path: str, cache_path: str = None):
    """Extract elements from PDF, using a cached JSON if available"""
    if cache_path is None:
        cache_path = file_path.replace(".pdf", "_elements.json")

    if os.path.exists(cache_path):
        print(f"✅ Found cached elements at {cache_path} — loading instead of re-parsing")
        elements = elements_from_json(cache_path)
        print(f"✅ Loaded {len(elements)} elements from cache")
        return elements

    print(f"📄 Partitioning document: {file_path} (this may take a while for hi_res)")
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )
    print(f"✅ Extracted {len(elements)} elements")

    elements_to_json(elements, filename=cache_path)
    print(f"💾 Cached elements to {cache_path} for next run")
    return elements


file_path = "docs/statement_sample1.pdf"
elements = partition_document(file_path)

✅ Found cached elements at docs/statement_sample1_elements.json — loading instead of re-parsing
✅ Loaded 20 elements from cache


In [44]:
def find_tables_camelot(pdf_path, min_accuracy=70, min_columns=2):
    """Run Camelot as a second opinion on table detection"""
    all_tables = {}
    for flavor in ["stream", "lattice"]:
        try:
            results = camelot.read_pdf(pdf_path, pages="all", flavor=flavor)
            for t in results:
                acc = t.parsing_report.get("accuracy", 0)
                if acc >= min_accuracy and t.df.shape[1] >= min_columns:
                    all_tables.setdefault(t.page, []).append({
                        "html": t.df.to_html(index=False, header=False),
                        "accuracy": acc
                    })
        except Exception as e:
            print(f"⚠️ Camelot {flavor} failed: {e}")
    return all_tables


def score_table_confidence(html, page_text_near_table=""):
    """Heuristically score whether a Camelot-detected 'table' is real or a false positive"""
    soup = BeautifulSoup(html, 'html.parser')
    cells = [c.get_text(strip=True) for c in soup.find_all(['td', 'th'])]
    cells = [c for c in cells if c]
    if not cells:
        return 0, ["no cell content"]

    score = 0
    reasons = []

    if re.search(r'\btable\s+\d+\b', page_text_near_table, re.IGNORECASE):
        score += 40
        reasons.append("+40: 'Table N' caption found nearby")

    numeric_ratio = sum(1 for c in cells if re.search(r'\d', c)) / len(cells)
    if numeric_ratio > 0.3:
        score += 25
        reasons.append(f"+25: numeric ratio {numeric_ratio:.0%}")
    elif numeric_ratio < 0.05:
        score -= 15
        reasons.append(f"-15: almost no numbers ({numeric_ratio:.0%})")

    avg_len = sum(len(c) for c in cells) / len(cells)
    if avg_len < 25:
        score += 20
        reasons.append(f"+20: short cells (avg {avg_len:.0f} chars)")
    elif avg_len > 60:
        score -= 25
        reasons.append(f"-25: long cells, likely prose (avg {avg_len:.0f} chars)")

    word_counts = Counter(cells)
    most_common_count = word_counts.most_common(1)[0][1] if word_counts else 0
    repetition_ratio = most_common_count / len(cells)
    if repetition_ratio > 0.15:
        score -= 30
        reasons.append(f"-30: high repetition ({repetition_ratio:.0%}) — likely figure/diagram text")

    return max(0, min(100, score)), reasons


def filter_camelot_tables(camelot_tables, elements, min_score=60):
    """Auto-classify each Camelot table as real or false-positive"""
    confirmed = {}
    page_text = {}
    for el in elements:
        pg = getattr(el.metadata, 'page_number', None)
        if pg:
            page_text.setdefault(pg, "")
            page_text[pg] += " " + el.text

    for page, tables_on_page in camelot_tables.items():
        for t in tables_on_page:
            score, reasons = score_table_confidence(t['html'], page_text.get(page, ""))
            if score >= min_score:
                confirmed.setdefault(page, []).append(t)

    print(f"✅ Confirmed real tables on pages: {sorted(confirmed.keys())}")
    return confirmed


camelot_tables_raw = find_tables_camelot(file_path)
confirmed_tables = filter_camelot_tables(camelot_tables_raw, elements)

# Only keep pages unstructured actually missed
unstructured_table_pages = {
    el.metadata.page_number for el in elements
    if type(el).__name__ == "Table" and hasattr(el.metadata, "page_number")
}
recovered_tables = {
    page: [t["html"] for t in tables]
    for page, tables in confirmed_tables.items()
    if page not in unstructured_table_pages
}
print(f"📋 Tables to inject (missed by unstructured): {list(recovered_tables.keys())}")

✅ Confirmed real tables on pages: []
📋 Tables to inject (missed by unstructured): []


In [45]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    print(f"✅ Created {len(chunks)} chunks")
    return chunks


chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 5 chunks


In [46]:
def crop_table_from_pdf(pdf_path, page_number, element, output_path, zoom=3, padding=8):
    """Crop the exact table region from the original PDF page — pixel-perfect"""
    try:
        coords = element.metadata.coordinates
        if not coords or not coords.points:
            return None

        doc = fitz.open(pdf_path)
        page = doc[page_number - 1]
        page_rect = page.rect  # actual PDF page size, in points

        # unstructured's coordinates may be in pixel space (hi_res rendering), not
        # PDF point space — scale to match the real page dimensions
        coord_system = coords.system
        system_width = getattr(coord_system, 'width', None)
        system_height = getattr(coord_system, 'height', None)

        if system_width and system_height:
            scale_x = page_rect.width / system_width
            scale_y = page_rect.height / system_height
        else:
            scale_x = scale_y = 1.0

        xs = [p[0] * scale_x for p in coords.points]
        ys = [p[1] * scale_y for p in coords.points]
        x0, x1 = min(xs) - padding, max(xs) + padding
        y0, y1 = min(ys) - padding, max(ys) + padding

        if x1 - x0 < 5 or y1 - y0 < 5:
            doc.close()
            return None

        x0, y0 = max(x0, 0), max(y0, 0)
        x1, y1 = min(x1, page_rect.width), min(y1, page_rect.height)

        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, clip=fitz.Rect(x0, y0, x1, y1))
        pix.save(output_path)
        doc.close()
        return output_path

    except Exception as e:
        print(f"     ⚠️ Could not crop table from PDF: {e}")
        return None


def clean_cell_text(cell):
    """Extract cell text, converting <sup>/<sub> tags to unicode"""
    sup_map = {'2': '²', '3': '³', '1': '¹'}
    text = ''
    for content in cell.contents:
        if getattr(content, 'name', None) == 'sup':
            raw = content.get_text()
            text += sup_map.get(raw, f'^{raw}')
        elif getattr(content, 'name', None) == 'sub':
            text += f"_{content.get_text()}"
        else:
            text += str(content) if isinstance(content, str) else content.get_text()
    return text.strip()


def html_table_to_image(html, output_path, title=None):
    """Fallback: render table from HTML if PDF cropping isn't available"""
    try:
        soup = BeautifulSoup(html, 'html.parser')
        rows = soup.find_all('tr')
        if not rows:
            return None
        data = [[clean_cell_text(c) for c in row.find_all(['td', 'th'])] for row in rows]
        header, body = data[0], data[1:]
        if not body or not header:
            return None
        col_count = len(header)
        body = [row + [''] * (col_count - len(row)) if len(row) < col_count else row[:col_count] for row in body]
        df = pd.DataFrame(body, columns=header)
    except Exception as e:
        print(f"     ⚠️ Could not parse table HTML: {e}")
        return None

    if df.empty:
        return None

    fig, ax = plt.subplots(figsize=(max(6, len(df.columns) * 1.4), max(1.5, len(df) * 0.5 + 1)))
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    tbl = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.6)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#4a4a4a')
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#f5f5f5')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    return output_path

In [47]:
def separate_content_types(chunk, chunk_id, pdf_path, recovered_tables):
    """Analyze content types AND persist images/tables to disk with paths"""
    content_data = {'text': chunk.text, 'tables': [], 'images': [], 'types': ['text'], 'page': None}

    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            if content_data['page'] is None and hasattr(element.metadata, 'page_number'):
                content_data['page'] = element.metadata.page_number

            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                table_idx = len(content_data['tables'])
                table_path = f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.html"
                with open(table_path, 'w', encoding='utf-8') as f:
                    f.write(f"<html><body>{table_html}</body></html>")

                img_path = f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.png"
                rendered = crop_table_from_pdf(pdf_path, content_data['page'], element, img_path)
                if rendered is None:
                    rendered = html_table_to_image(table_html, img_path, title=f"Table (page {content_data['page']})")

                content_data['tables'].append({"html": table_html, "path": table_path, "image_path": rendered})

            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    img_b64 = element.metadata.image_base64
                    img_idx = len(content_data['images'])
                    img_out_path = f"extracted_data/images/chunk_{chunk_id}_img_{img_idx}.png"
                    with open(img_out_path, 'wb') as f:
                        f.write(base64.b64decode(img_b64))
                    content_data['images'].append({"base64": img_b64, "path": img_out_path})

        # Inject Camelot-recovered tables for this chunk's page (unstructured missed these)
        if content_data['page'] in recovered_tables:
            for html in recovered_tables[content_data['page']]:
                content_data['types'].append('table')
                table_idx = len(content_data['tables'])
                table_path = f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.html"
                with open(table_path, 'w', encoding='utf-8') as f:
                    f.write(f"<html><body>{html}</body></html>")
                img_path = f"extracted_data/tables/chunk_{chunk_id}_table_{table_idx}.png"
                rendered = html_table_to_image(html, img_path, title=f"Table (page {content_data['page']}, recovered)")
                content_data['tables'].append({"html": html, "path": table_path, "image_path": rendered})
            del recovered_tables[content_data['page']]

    content_data['types'] = list(set(content_data['types']))
    return content_data

In [48]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    try:
        model_name = "qwen/qwen3.6-27b" if images else "openai/gpt-oss-120b"
        llm = ChatGroq(model_name=model_name, temperature=0)

        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}
        """

        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
            prompt_text += """
IMPORTANT: Scan the TEXT CONTENT above for any explicit table number, label, or caption
(e.g. "Table 3"). If found, state it verbatim as the FIRST LINE of your description in
the format: "This is Table X: <topic>". Always include the exact table number if present.
"""

        prompt_text += """
        YOUR TASK:
        Generate a comprehensive, searchable description covering key facts, main topics,
        questions this content could answer, visual content analysis, and alternative search terms.

        SEARCHABLE DESCRIPTION:"""

        if images:
            message_content = [{"type": "text", "text": prompt_text}]
            for image_base64 in images:
                message_content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}})
        else:
            message_content = prompt_text

        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content

    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [49]:
def save_processed_chunks(documents, cache_path):
    """Save processed_chunks (LangChain Documents) to disk as JSON"""
    data = [
        {"page_content": doc.page_content, "metadata": doc.metadata}
        for doc in documents
    ]
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved {len(documents)} processed chunks to {cache_path}")


def load_processed_chunks(cache_path):
    """Load processed_chunks back into LangChain Documents"""
    with open(cache_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    documents = [Document(page_content=item["page_content"], metadata=item["metadata"]) for item in data]
    print(f"✅ Loaded {len(documents)} processed chunks from cache")
    return documents


def summarise_chunks(chunks, pdf_path, recovered_tables, cache_path=None):
    """Process all chunks with AI Summaries, saving images/tables to disk.
    Uses a cached JSON if available, to avoid re-running Groq summarization."""

    if cache_path is None:
        cache_path = pdf_path.replace(".pdf", "_processed_chunks.json")

    if os.path.exists(cache_path):
        print(f"✅ Found cached processed chunks at {cache_path} — skipping re-summarization")
        return load_processed_chunks(cache_path)

    print("🧠 Processing chunks with AI Summaries...")
    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        print(f"   Processing chunk {i+1}/{total_chunks}")
        content_data = separate_content_types(chunk, chunk_id=i, pdf_path=pdf_path, recovered_tables=recovered_tables)
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        table_htmls = [t["html"] for t in content_data['tables']]
        image_b64s = [img["base64"] for img in content_data['images']]

        if table_htmls or image_b64s:
            enhanced_content = create_ai_enhanced_summary(content_data['text'], table_htmls, image_b64s)
        else:
            enhanced_content = content_data['text']

        doc = Document(
            page_content=enhanced_content,
            metadata={
                "chunk_id": i,
                "page": content_data['page'] or 0,
                "raw_text": content_data['text'][:2000],
                "table_paths": json.dumps([t["path"] for t in content_data['tables']]),
                "table_image_paths": json.dumps([t["image_path"] for t in content_data['tables'] if t.get("image_path")]),
                "image_paths": json.dumps([img["path"] for img in content_data['images']]),
                "has_table": bool(content_data['tables']),
                "has_image": bool(content_data['images']),
            }
        )
        langchain_documents.append(doc)

    print(f"✅ Processed {len(langchain_documents)} chunks")

    save_processed_chunks(langchain_documents, cache_path)
    return langchain_documents


processed_chunks = summarise_chunks(chunks, pdf_path=file_path, recovered_tables=recovered_tables)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/5
     Types found: ['image', 'text']
     Tables: 0, Images: 1
   Processing chunk 2/5
     Types found: ['table', 'text']
     Tables: 1, Images: 0
   Processing chunk 3/5
     Types found: ['text']
     Tables: 0, Images: 0
   Processing chunk 4/5
     Types found: ['table', 'text']
     Tables: 1, Images: 0
   Processing chunk 5/5
     Types found: ['text']
     Tables: 0, Images: 0
✅ Processed 5 chunks
💾 Saved 5 processed chunks to docs/statement_sample1_processed_chunks.json


In [55]:
class NomicEmbeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        return super().embed_documents([f"search_document: {t}" for t in texts])
    def embed_query(self, text):
        return super().embed_query(f"search_query: {text}")


def create_vector_store(documents, persist_directory="Statement_pdf_db/chroma_db"):
    """Create and persist ChromaDB vector store, or load it if already created"""
    embedding_model = NomicEmbeddings(model_name="nomic-ai/nomic-embed-text-v1.5", model_kwargs={"trust_remote_code": True})
    db_file = os.path.join(persist_directory, "chroma.sqlite3")

    if os.path.exists(db_file):
        print(f"✅ Vector store already created at {persist_directory} — loading existing DB")
        return Chroma(persist_directory=persist_directory, embedding_function=embedding_model, collection_metadata={"hnsw:space": "cosine"})

    print("🔮 Creating embeddings and storing in ChromaDB...")
    vectorstore = Chroma.from_documents(
        documents=documents, embedding=embedding_model, persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}, ids=[f"chunk_{i}" for i in range(len(documents))]
    )
    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore


db = create_vector_store(processed_chunks)

<All keys matched successfully>


🔮 Creating embeddings and storing in ChromaDB...
✅ Vector store created and saved to Statement_pdf_db/chroma_db


In [56]:
def smart_retrieve(query, db, processed_chunks, k=5):
    """Retrieve normally, but force-include the specific table chunk if query names a table number"""
    results = db.similarity_search(query, k=k)
    seen_ids = {d.metadata.get("chunk_id") for d in results}

    table_match = re.search(r'table\s*(\d+)', query, re.IGNORECASE)
    if table_match:
        target_pattern = re.compile(rf'\btable\s*{table_match.group(1)}\b', re.IGNORECASE)
        for doc in processed_chunks:
            if not doc.metadata.get("has_table") or doc.metadata.get("chunk_id") in seen_ids:
                continue
            searchable = doc.page_content + " " + doc.metadata.get("raw_text", "")
            if target_pattern.search(searchable):
                results.append(doc)
                seen_ids.add(doc.metadata.get("chunk_id"))

    return results

In [57]:
def generate_final_answer(chunks, query):
    """Generate final answer + structured source list"""
    sources = []
    has_images = False

    try:
        prompt_text = f"Based on the following documents, please answer this question: {query}\n\nCONTENT TO ANALYZE:\n"

        for i, chunk in enumerate(chunks):
            meta = chunk.metadata
            prompt_text += f"--- Document {i+1} (page {meta.get('page')}) ---\nTEXT:\n{meta.get('raw_text', chunk.page_content)}\n\n"

            table_paths = json.loads(meta.get("table_paths", "[]"))
            table_image_paths = json.loads(meta.get("table_image_paths", "[]"))
            image_paths = json.loads(meta.get("image_paths", "[]"))

            if table_paths:
                prompt_text += "TABLES:\n"
                for p in table_paths:
                    with open(p, 'r', encoding='utf-8') as f:
                        prompt_text += f.read() + "\n\n"

            source_entry = {"chunk_id": meta.get("chunk_id"), "page": meta.get("page"), "type": "text", "preview": chunk.page_content[:150], "paths": []}
            if table_paths:
                source_entry["type"] = "table"
                source_entry["paths"].extend(table_image_paths if table_image_paths else table_paths)
            if image_paths:
                has_images = True
                source_entry["type"] = "image" if not table_paths else "table+image"
                source_entry["paths"].extend(image_paths)

            sources.append(source_entry)
            prompt_text += "\n"

        prompt_text += '\nPlease provide a clear, comprehensive answer using the text, tables, and images above. If the documents don\'t contain sufficient information, say "I don\'t have enough information to answer that question based on the provided documents."\n\nANSWER:'

        model_name = "qwen/qwen3.6-27b" if has_images else "openai/gpt-oss-120b"
        llm = ChatGroq(model_name=model_name, temperature=0)

        if has_images:
            message_content = [{"type": "text", "text": prompt_text}]
            for chunk in chunks:
                for path in json.loads(chunk.metadata.get("image_paths", "[]")):
                    with open(path, 'rb') as f:
                        img_b64 = base64.b64encode(f.read()).decode('utf-8')
                    message_content.append({"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}})
        else:
            message_content = prompt_text

        response = llm.invoke([HumanMessage(content=message_content)])
        answer_text = response.content

        if "don't have enough information" in answer_text.lower():
            sources = []

        return answer_text, sources

    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer.", sources

In [60]:
def render_table_html(raw_html):
    return f"""
    <style>
        .rag-table-wrapper {{ font-family: -apple-system, sans-serif; font-size: 13px; overflow-x: auto; margin: 10px 0; }}
        .rag-table-wrapper table {{ border-collapse: collapse; width: 100%; }}
        .rag-table-wrapper th, .rag-table-wrapper td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
        .rag-table-wrapper th {{ background-color: #f0f0f0; font-weight: 600; }}
        .rag-table-wrapper tr:nth-child(even) {{ background-color: #fafafa; }}
    </style>
    <div class="rag-table-wrapper">{raw_html}</div>
    """


def display_query_result(query, answer, sources):
    print(f"❓ Query: {query}\n")
    print(f"💡 Answer:\n{answer}\n")
    print(f"📚 Sources ({len(sources)}):\n")

    output_area = widgets.Output()

    def make_click_handler(source):
        def handler(b):
            with output_area:
                clear_output(wait=True)
                print(f"--- chunk {source['chunk_id']} | page {source['page']} | type: {source['type']} ---\n")
                if not source['paths']:
                    print(source['preview'])
                for path in source['paths']:
                    if path.endswith(('.png', '.jpg', '.jpeg')):
                        display(IPImage(filename=path))
                    elif path.endswith('.html'):
                        with open(path, 'r', encoding='utf-8') as f:
                            display(HTML(render_table_html(f.read())))
        return handler

    buttons = [widgets.Button(description=f"[{i+1}] page {s['page']} • {s['type']}", layout=widgets.Layout(width='auto')) for i, s in enumerate(sources)]
    for btn, source in zip(buttons, sources):
        btn.on_click(make_click_handler(source))

    display(widgets.HBox(buttons))
    display(output_area)

In [61]:
query = "What is the name of the bank?"

retrieved_chunks = smart_retrieve(query, db, processed_chunks, k=5)
answer, sources = generate_final_answer(retrieved_chunks, query)
display_query_result(query, answer, sources)

❓ Query: What is the name of the bank?

💡 Answer:

<think>
The user wants to identify the name of the bank from the provided documents.

1.  **Analyze Document 1:** The text at the very top says "e/ INO eat wn Commerce Bank Member FDIC". This looks like a garbled OCR of a logo. The words "Commerce Bank" are clearly visible.
2.  **Analyze the Image:** The image provided shows a logo with a globe icon and the text "Commerce Bank" next to it. Below that, it says "Member FDIC". The image also contains cropped parts of the logo showing "Com", "nce", "Ban", "k", and "Member FDIC".
3.  **Synthesize findings:** Both the text in Document 1 and the provided image clearly state the name of the bank.

**Conclusion:** The bank name is Commerce Bank.
</think>

Based on the text in Document 1 and the provided image, the name of the bank is **Commerce Bank**.

📚 Sources (5):



Output()